# Phase 5-B3-C｜第一次真實歷史市場研究（RA-IBKR_TSM-SMA520_P5B3C-0001）

**只由研究負責人 hh4832 本人執行。** 這份 Notebook 會以既有的 Phase 5 治理流程，執行唯一 1 個 batch：

`A2 Proposal → Research Request → Binding → Task 核准 → REAL_GATED 啟用閘門 A01–A10 → 原子額度保留 → 既有 ResearchOnlyRunner + Core b16179a → 證據驗證 → 評估 → 封存 → 結清（CONSUMED）`

研究目的是 `PIPELINE_ACCEPTANCE_ONLY`：在已評估過的 DEV 期間驗證整條流程，**不是** SMA 5/20 有效性的新統計證據。過程中不讀 Holdout，也不呼叫券商。

**整個執行分兩次 Run All，中間由你在 GitHub 留兩則核可留言：**

| 次序 | 第 1 格設定 | 做什麼 | 消耗額度？ |
|---|---|---|---|
| 第 1 次 | `STAGE = "PREPARE"` | 產生 Research Request，寫到 Drive；印出 **Request Hash、Task ID 與兩行核可文字** | 否 |
| 你 | — | 確認 hash／Task ID 後，把兩行文字**分別**貼成兩則留言（PR #19 的 Conversation），然後複製兩則留言的連結 | 否 |
| 第 2 次 | `STAGE = "EXECUTE"`，填入兩個留言連結，`I_CONFIRM_SINGLE_BATCH = True` | 以 Production GitHub Verifier 驗證兩則留言 → binding → 核准 → 執行 1 次 → 封存與結清 → 推送紀錄到新分支 | **是，1/1** |

**PAT 權限**：Colab Secrets 的 `GITHUB_TOKEN` 需要以下權限，只授權 `hh4832/strategy-research-agent`：

- Contents：Read and write
- Issues：Read-only
- Pull requests：Read-only

PAT 只存在於這個 runtime 的記憶體中，不會被印出，也不會寫入檔案或 commit。

**規則**：

- 兩次 Run All 都要使用全新 runtime。
- 第 4 格（執行）失敗時，**不要重跑**。Drive 上的 `CLAIM.json` 會阻止任何第二次嘗試，額度依規約保持已消耗，交由研究負責人對帳。
- 核可留言的有效期是 PREPARE 後 48 小時；授權本身到 2026-11-17 11:54（台北時間）為止。


In [ ]:
# 第 1 格：所有人工核可與輸入集中在這裡；任何一項不成立立即停止（fail fast）
STAGE = "PREPARE"            # 第 1 次 "PREPARE"；留言貼好後第 2 次改成 "EXECUTE"
BIND_COMMENT_URL = ""        # EXECUTE：BIND_AUTHORIZATION 那則留言的連結（…#issuecomment-…）
TASK_COMMENT_URL = ""        # EXECUTE：APPROVE_TASK 那則留言的連結
I_CONFIRM_SINGLE_BATCH = False   # EXECUTE：確認要消耗唯一 1 個 batch 時改成 True

from pathlib import Path
import json, os, re, subprocess, sys, tempfile, urllib.request, hashlib, shutil
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

REPO = "hh4832/strategy-research-agent"
BRANCH = "feature/phase5b3c-execution"
PIN = "df040bb29508a7ff70e141b833b145c5cfe83cdf"                       # 已審查的執行程式碼；之後只允許 docs/、notebooks/ 變動（Version Sync）
CORE_SHA = "b16179a7cc077c36046574769a32c367fa476cae"
ID = "RA-IBKR_TSM-SMA520_P5B3C-0001"
EXECUTOR = "colab-hh4832-exec-01"
APPROVER = "hh4832"
SNAPSHOT_ID = "TSM_DEV_1M_20240529_20240628_18350b3adba8e28a7e5e"
MANIFEST_SHA = "8b6fc838b72ae3015f6eb418c3a77115e53a86f185e05c2249e9dcd58192cfc5"
PARQUET_SHA = "239d5741d65cc0c7118f618a5ebb7b1dd946f385d843cae151a24edc1e0a8452"
SNAPSHOT = Path("/content/drive/MyDrive/00Quant_Research/data/development_snapshots") / SNAPSHOT_ID
DRIVE_B3C = Path("/content/drive/MyDrive/00Quant_Research/phase5b3c") / ID     # 執行者的持久狀態（O_EXCL）
PREP = DRIVE_B3C / "prepare"
WORK = Path("/content/phase5b3c_repo")
RESULT_BRANCH = "research/phase5b3c-result-p5b3c-0001"
URL_RE = re.compile(r"^https://github\.com/hh4832/strategy-research-agent/(pull|issues)/\d+#issuecomment-\d+$")
TPE = ZoneInfo("Asia/Taipei")
assert sys.version_info >= (3, 11), "需要 Python 3.11 以上"
assert STAGE in ("PREPARE", "EXECUTE"), "STAGE 只能是 PREPARE 或 EXECUTE"
assert re.fullmatch(r"[0-9a-f]{40}", PIN)
# (a) 全新 runtime
assert not WORK.exists(), "請 Runtime → Disconnect and delete runtime 後重新 Run All（全新 runtime）"
# (b) Drive 授權（人工核可視窗）
from google.colab import drive, userdata
drive.mount("/content/drive")
assert (SNAPSHOT / "manifest.json").is_file(), f"Drive 上找不到 DEV snapshot：{SNAPSHOT}"
# (c) Colab Secret + GitHub 身分（token 只在記憶體）
TOKEN = userdata.get("GITHUB_TOKEN")
assert TOKEN, "請在 Colab Secrets 設定 GITHUB_TOKEN 並開啟此 Notebook 的存取權"
def gh(path):
    req = urllib.request.Request("https://api.github.com" + path,
                                 headers={"Authorization": "Bearer " + TOKEN, "Accept": "application/vnd.github+json"})
    with urllib.request.urlopen(req, timeout=25) as r:
        return json.load(r)
login = gh("/user")["login"]
assert login.lower() == APPROVER, "GitHub token 所屬帳號不是研究負責人 hh4832，停止"
# (d) 依階段檢查輸入與 Drive 狀態
if STAGE == "PREPARE":
    assert not PREP.exists(), f"Drive 已有 PREPARE 結果（{PREP}）；不重新產生。請改用 STAGE=EXECUTE"
else:
    assert I_CONFIRM_SINGLE_BATCH is True, "EXECUTE 需要 I_CONFIRM_SINGLE_BATCH = True"
    assert URL_RE.match(BIND_COMMENT_URL) and URL_RE.match(TASK_COMMENT_URL), "留言連結格式不符（需含 #issuecomment-）"
    assert BIND_COMMENT_URL != TASK_COMMENT_URL, "兩個核可必須是兩則不同的留言"
    assert (PREP / "PREPARE_MANIFEST.json").is_file(), "Drive 上沒有 PREPARE 結果；請先跑 STAGE=PREPARE"
    assert not (DRIVE_B3C / "CLAIM.json").exists(), "Drive 已有 CLAIM.json：已有過執行嘗試，依規約不得重跑（交研究負責人對帳）"
    for u in (BIND_COMMENT_URL, TASK_COMMENT_URL):     # 及早確認 token 能讀留言（Issues／Pull requests 讀取權限）
        gh(f"/repos/{REPO}/issues/comments/{u.rsplit('-', 1)[1]}")
print(f"第 1 格 PASS：STAGE={STAGE}｜GitHub 身分 {login}｜Drive 已掛載｜全新 runtime｜executor {EXECUTOR}")

def run(*cmd, cwd=None, env=None, check=True):
    r = subprocess.run(cmd, cwd=cwd, env=env, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{cmd[:3]} 失敗（exit {r.returncode}）：{r.stderr[-1500:]}")
    return r
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""): h.update(c)
    return h.hexdigest()


In [ ]:
# 第 2 格：乾淨 clone、Version Sync、安裝 Core b16179a、授權狀態（不寫任何帳本）
run("git", "clone", "--quiet", f"https://github.com/{REPO}.git", str(WORK))
ref = "origin/main" if run("git", "merge-base", "--is-ancestor", PIN, "origin/main", cwd=WORK, check=False).returncode == 0 \
    else "origin/" + BRANCH
run("git", "checkout", "--quiet", "--detach", ref, cwd=WORK)
HEAD = run("git", "rev-parse", "HEAD", cwd=WORK).stdout.strip()
assert run("git", "merge-base", "--is-ancestor", PIN, HEAD, cwd=WORK, check=False).returncode == 0, "PIN 不在歷史中"
drift = [f for f in run("git", "diff", "--name-only", PIN, HEAD, cwd=WORK).stdout.split() if f]
assert all(f.startswith(("docs/", "notebooks/")) for f in drift), f"Version Sync 失敗：PIN 之後有程式／帳本變動 {drift}"
assert run("git", "status", "--porcelain", cwd=WORK).stdout.strip() == ""
run(sys.executable, "-m", "pip", "install", "--quiet", "-e", str(WORK) + "[research]", "pyarrow",
    f"market-research-core @ git+https://github.com/hh4832/market-research-core@{CORE_SHA}")
sys.path.insert(0, str(WORK / "src"))
from strategy_agent.readiness import b3c
from strategy_agent.execution import b3c_run as R
from strategy_agent.adapters.mrc_phase4a import core_phase4a
assert b3c.ROOT == WORK and R.ROOT == WORK
assert core_phase4a()[0].core_code_identity()["core_git_sha"] == CORE_SHA, "安裝的 Core 不是 b16179a"
st = b3c.formal_authorization_state()
assert st["state"] == "GRANTED_UNUSED", f"授權狀態不是 GRANTED_UNUSED：{st}"
assert datetime.now(TPE) < datetime.fromisoformat(st["validity"]["expires_at"]), "授權已過期"
assert b3c.expected_eligible() == [ID]
pf = b3c.preflight()
assert pf["preparation"] == "AUTHORIZED_AWAITING_GITHUB_APPROVALS" and not pf["preparation_problems"], pf["preparation_problems"]
assert (sha256(SNAPSHOT / "manifest.json"), json.loads((SNAPSHOT / "manifest.json").read_text())["parquet_sha256"]) == \
    (MANIFEST_SHA, PARQUET_SHA), "Drive 上的 snapshot 與釘選不符"
print("第 2 格 PASS：HEAD", HEAD[:7], f"（{ref}，PIN {PIN[:7]}）｜Core b16179a｜授權 GRANTED_UNUSED 0/1，到期",
      st["validity"]["expires_at"], "｜snapshot 釘選相符")


In [ ]:
# 第 3 格：PREPARE（不消耗額度）— 產生 Request 並印出兩行核可文字
if STAGE == "PREPARE":
    m = R.prepare(PREP, agent_commit=HEAD)
    print("Source Request Hash（BIND_AUTHORIZATION 用）：", m["source_request_sha256"])
    print("Bound Request Hash（APPROVE_TASK 用）     ：", m["bound_request_sha256"])
    print("Task ID：", m["task_id"], "｜留言有效至（UTC）：", m["expires_utc"])
    print("\n請在 https://github.com/" + REPO + "/pull/19 的 Conversation 貼兩則「不同」的留言（每則只貼一行）：\n")
    print("① 第一則（BIND_AUTHORIZATION）：\n" + m["BIND_AUTHORIZATION"] + "\n")
    print("② 第二則（APPROVE_TASK）：\n" + m["APPROVE_TASK"] + "\n")
    print("貼好後：點各留言右上角 … → Copy link，填入第 1 格；STAGE 改 EXECUTE、I_CONFIRM_SINGLE_BATCH 改 True，")
    print("然後 Disconnect and delete runtime，再 Run All。PREPARE 結果已寫入 Drive：", PREP)
else:
    print("STAGE=EXECUTE：略過 PREPARE")


In [ ]:
# 第 4 格：EXECUTE（消耗唯一 1 個 batch）— 不得重跑
EXEC = None
if STAGE == "EXECUTE":
    from strategy_agent.approval import production_verifier
    os.environ["GITHUB_TOKEN"] = TOKEN                       # 只給 Production GitHub Verifier 讀留言；不寫入任何檔案
    stamp = datetime.now(TPE).strftime("%Y%m%d_%H%M%S")
    verifier = production_verifier()
    assert verifier.production, "必須是 Production（live GitHub）verifier"
    backend = R.production_backend(root=WORK, snapshot_root=SNAPSHOT, session_dir=DRIVE_B3C / "sessions" / stamp,
                                   git_commit=HEAD, context={"agent_checkout_clean": True, "notebook": "PHASE5B3C_EXECUTION_v1",
                                                             "version_sync": {"pin": PIN, "head": HEAD, "ref": ref}})
    try:
        EXEC = R.execute(PREP, bind_url=BIND_COMMENT_URL, task_url=TASK_COMMENT_URL, verifier=verifier, backend=backend,
                         work_dir=Path("/content/phase5b3c_run"), claim_dir=DRIVE_B3C)
    except R.B3CStopped as e:
        claimed = (DRIVE_B3C / "CLAIM.json").exists()
        print("停止：", e.code, "｜CLAIM 存在：", claimed, "（存在代表可能已消耗；不存在代表未消耗）")
        print(json.dumps(e.detail, ensure_ascii=False, default=str)[:3000])
        raise
    finally:   # 無論成敗，立即把本機執行狀態完整複製到 Drive（新目錄，不覆寫）
        if Path("/content/phase5b3c_run").exists():
            shutil.copytree("/content/phase5b3c_run", DRIVE_B3C / "runs" / stamp)
        recs = DRIVE_B3C / "runs" / stamp / "repo_records"
        for f in run("git", "status", "--porcelain", "--untracked-files=all", cwd=WORK).stdout.splitlines():
            (recs / f[3:]).parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(WORK / f[3:], recs / f[3:])
    ACC = R.acceptance(EXEC, governance_root=WORK, cloud=True, production=True)
    EVID = WORK / "docs/evidence/phase5b3c" / f"{stamp}_{HEAD[:7]}"
    IDX = R.evidence_bundle(EXEC, PREP, EVID, acceptance_doc=ACC)
    shutil.copytree(EVID, DRIVE_B3C / "evidence" / EVID.name)
    print("狀態：", EXEC["status"]["state"], "｜budget：", EXEC["status"].get("budget_state"), "｜settled：", EXEC["settled"])
    print("B3C gates：", {k: v["status"] for k, v in ACC["gates"].items()})
    print("Verdict：", ACC["verdict"], "（B3C-10 需研究負責人書面接受結果）")
    rd = EVID / "task_archive/research_decision.json"
    if rd.is_file():
        print(json.dumps(json.loads(rd.read_text()), indent=1, ensure_ascii=False)[:2000])
    rp = EVID / "runner_archive/research_report.md"
    if rp.is_file():
        print(rp.read_text()[:4000])
else:
    print("STAGE=PREPARE：略過 EXECUTE")


In [ ]:
# 第 5 格：把治理紀錄與證據推到新分支（不合併；由你本人開 PR 並合併 = B3C-10 結清紀錄）
if STAGE == "EXECUTE":
    changed = sorted(f[3:] for f in run("git", "status", "--porcelain", "--untracked-files=all", cwd=WORK).stdout.splitlines())
    assert changed and all(f.startswith(("registry/", "docs/evidence/phase5b3c/")) for f in changed), f"意外變更：{changed}"
    assert run("git", "ls-remote", "--heads", "origin", RESULT_BRANCH, cwd=WORK).stdout.strip() == "", "結果分支已存在；不覆寫"
    run("git", "checkout", "--quiet", "-b", RESULT_BRANCH, cwd=WORK)
    run("git", "add", "--", *changed, cwd=WORK)
    run("git", "-c", f"user.name={APPROVER}", "-c", f"user.email={APPROVER}@users.noreply.github.com", "commit", "-q",
        "-m", f"research: Phase 5-B3-C execution records for {ID} ({EXEC['status']['state']}, budget "
              f"{EXEC['status'].get('budget_state')}, verdict {ACC['verdict']})", cwd=WORK)
    with tempfile.TemporaryDirectory() as d:
        ask = Path(d) / "askpass.sh"
        ask.write_text("#!/bin/sh\ncase \"$1\" in *Username*) printf '%s' \"$GIT_USERNAME\";; *) printf '%s' \"$GIT_PASSWORD\";; esac\n")
        ask.chmod(0o700)
        env = dict(os.environ, GIT_ASKPASS=str(ask), GIT_TERMINAL_PROMPT="0", GIT_USERNAME="x-access-token", GIT_PASSWORD=TOKEN)
        res = subprocess.run(["git", "push", "origin", "HEAD:refs/heads/" + RESULT_BRANCH], cwd=WORK, env=env, text=True,
                             capture_output=True)
        env["GIT_PASSWORD"] = ""
    if res.returncode != 0:
        raise RuntimeError(f"push 未完成（exit {res.returncode}）；紀錄已在 Drive：{DRIVE_B3C}。請勿重跑第 4 格")
    print("已推送", len(changed), "個檔案到", RESULT_BRANCH, "｜commit", run("git", "rev-parse", "HEAD", cwd=WORK).stdout.strip())
    print("開 PR：https://github.com/" + REPO + "/compare/main..." + RESULT_BRANCH + "?expand=1")
    print("Drive 完整紀錄：", DRIVE_B3C)
else:
    print("STAGE=PREPARE：完成。請到 GitHub 貼兩則核可留言後再執行 EXECUTE。")


## 驗收

EXECUTE 成功時，第 4 格會依證據印出以下結果：

- B3C-01～09 全部 PASS。
- B3C-10 為 `PENDING_RESEARCH_LEAD_DECISION`。
- verdict 為 `EXECUTED_PENDING_RESULT_ACCEPTANCE`。

第 5 格會把以下紀錄推送到 `research/phase5b3c-result-p5b3c-0001` 分支：

- binding 紀錄與證據
- budget ledger
- 授權使用紀錄（1/1）
- `docs/evidence/phase5b3c/<run>/`，內容包括：
  - PREPARE 檔案
  - execution record
  - task archive
  - runner archive 摘要（不含市場資料）
  - B3C acceptance
  - ARCHIVE_INDEX

完整的本機狀態也會同步存在 Drive 的 `00Quant_Research/phase5b3c/RA-IBKR_TSM-SMA520_P5B3C-0001/`。

任何 FAIL 都代表 NO_GO，batch 保持已消耗，不會重置。PASS 只代表這條治理流程在真實資料上完整跑過一次，與 SMA 5/20 是否獲利無關。
